In [64]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [65]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Brute Force\\combined_brute_force.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "brute_force_bayesian.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Brute Force\combined_brute_force.csv | Total rows: 7238
Train: 4342 | Val: 1448 | Test: 1448


In [66]:
from sklearn.preprocessing import KBinsDiscretizer

X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# BN requires discrete features — bin continuous values into categories
discretizer = KBinsDiscretizer(n_bins=6, encode="ordinal", strategy="quantile")
X_train_disc = pd.DataFrame(discretizer.fit_transform(X_train), columns=X_train.columns).astype(int)
X_val_disc   = pd.DataFrame(discretizer.transform(X_val),   columns=X_val.columns).astype(int)
X_test_disc  = pd.DataFrame(discretizer.transform(X_test),  columns=X_test.columns).astype(int)

# Combine with label for pgmpy
train_data = X_train_disc.copy()
train_data["label"] = y_train.values

C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(
C:\Users\johfr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\preprocessing\

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

TOP_K = 20 # tuneable parameter for how many features to keep

selector = SelectKBest(mutual_info_classif, k=TOP_K)
selector.fit(X_train_disc, y_train)

selected_cols = X_train_disc.columns[selector.get_support()].tolist()
print(f"Selected features: {selected_cols}")

X_train_disc = X_train_disc[selected_cols]
X_val_disc   = X_val_disc[selected_cols]
X_test_disc  = X_test_disc[selected_cols]

# Rebuild train_data with only selected features
train_data = X_train_disc.copy()
train_data["Label"] = y_train.values

Selected features: ['Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Flow Packets/s', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Std', 'Bwd IAT Std', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Fwd Act Data Pkts']


In [68]:
from pgmpy.estimators import HillClimbSearch
from pgmpy.estimators import BIC
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator

print("Learning structure...")
hc     = HillClimbSearch(train_data)
best_model_structure = hc.estimate(scoring_method=BIC(train_data), max_indegree=3)


print("Edges found:", best_model_structure.edges())

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Flow Duration': 'N', 'Total Fwd Packet': 'N', 'Total Bwd packets': 'N', 'Total Length of Fwd Packet': 'N', 'Fwd Packet Length Max': 'N', 'Fwd Packet Length Min': 'N', 'Flow Packets/s': 'N', 'Flow IAT Max': 'N', 'Fwd IAT Total': 'N', 'Fwd IAT Std': 'N', 'Bwd IAT Std': 'N', 'Fwd Header Length': 'N', 'Bwd Header Length': 'N', 'Fwd Packets/s': 'N', 'Packet Length Max': 'N', 'Packet Length Mean': 'N', 'PSH Flag Count': 'N', 'ACK Flag Count': 'N', 'Average Packet Size': 'N', 'Fwd Act Data Pkts': 'N', 'Label': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Flow Duration': 'N', 'Total Fwd Packet': 'N', 'Total Bwd packets': 'N', 'Total Length of Fwd Packet': 'N', 'Fwd Packet Length Max': 'N', 'Fwd Packet Length Min': 'N', 'Flow Packets/s': 'N', 'Flow IAT Max': 'N', 'Fwd IAT Total': 'N', 'Fwd IAT Std': 'N', 'Bwd IAT Std': 'N',

Learning structure...


  0%|          | 41/1000000 [00:01<7:56:48, 34.95it/s] 

Edges found: [('Flow Duration', 'Fwd IAT Total'), ('Flow Duration', 'Label'), ('Total Fwd Packet', 'Fwd IAT Std'), ('Total Fwd Packet', 'Fwd Act Data Pkts'), ('Total Fwd Packet', 'Total Bwd packets'), ('Total Bwd packets', 'Bwd IAT Std'), ('Total Bwd packets', 'Fwd Packets/s'), ('Fwd Packet Length Max', 'Total Length of Fwd Packet'), ('Fwd Packet Length Max', 'Packet Length Max'), ('Fwd Packet Length Max', 'Label'), ('Fwd Packet Length Min', 'Fwd Packet Length Max'), ('Fwd Packet Length Min', 'Average Packet Size'), ('Fwd Packet Length Min', 'ACK Flag Count'), ('Fwd Packet Length Min', 'PSH Flag Count'), ('Flow Packets/s', 'Fwd Packets/s'), ('Flow Packets/s', 'Flow IAT Max'), ('Flow Packets/s', 'Fwd IAT Total'), ('Flow Packets/s', 'Fwd Header Length'), ('Flow IAT Max', 'Flow Duration'), ('Flow IAT Max', 'Bwd Header Length'), ('Flow IAT Max', 'Fwd IAT Std'), ('Fwd IAT Total', 'Fwd Header Length'), ('Fwd IAT Total', 'Total Fwd Packet'), ('Fwd IAT Std', 'Packet Length Mean'), ('Fwd IAT St

In [69]:
from pgmpy.models import DiscreteBayesianNetwork

bn_model = DiscreteBayesianNetwork(best_model_structure.edges())
bn_model.fit(train_data, estimator=BayesianEstimator, prior_type="BDeu")
print("Model fitted.")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'Flow Duration': 'N', 'Total Fwd Packet': 'N', 'Total Bwd packets': 'N', 'Total Length of Fwd Packet': 'N', 'Fwd Packet Length Max': 'N', 'Fwd Packet Length Min': 'N', 'Flow Packets/s': 'N', 'Flow IAT Max': 'N', 'Fwd IAT Total': 'N', 'Fwd IAT Std': 'N', 'Bwd IAT Std': 'N', 'Fwd Header Length': 'N', 'Bwd Header Length': 'N', 'Fwd Packets/s': 'N', 'Packet Length Max': 'N', 'Packet Length Mean': 'N', 'PSH Flag Count': 'N', 'ACK Flag Count': 'N', 'Average Packet Size': 'N', 'Fwd Act Data Pkts': 'N', 'Label': 'N'}


Model fitted.


In [70]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(bn_model)

preds = []
for _, row in X_test_disc.iterrows():
    evidence = row.to_dict()
    result   = infer.map_query(variables=["Label"], evidence=evidence, show_progress=False)
    preds.append(result["Label"])

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.76      0.75      0.76       724
      Attack       0.75      0.77      0.76       724

    accuracy                           0.76      1448
   macro avg       0.76      0.76      0.76      1448
weighted avg       0.76      0.76      0.76      1448

[[542 182]
 [167 557]]
